In [ ]:
!pip -q install -U transformers accelerate sentencepiece datasets \
    rouge-score sacrebleu sentence-transformers scikit-learn pandas tqdm bitsandbytes


In [ ]:
import json, re, random, itertools, textwrap
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score
from rouge_score import rouge_scorer
import sacrebleu
from sentence_transformers import SentenceTransformer, util as st_util
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# Key inside each record that (once available) holds free-text gold reasoning.
# Falls back to concatenated gold evidence sentences if this key is absent.
GOLD_REASONING_KEY = "gold_reasoning"

# Lightweight, free, open-weight instruction model for the LLM baselines.
# Swap for a larger one (e.g. "Qwen/Qwen2.5-3B-Instruct") if you have more compute,
# or a CPU-only fallback ("google/flan-t5-base") if no GPU is available.
LLM_MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

# Legal-domain encoder for the BERT classifier baseline (free, open-weight).
BERT_MODEL_NAME = "nlpaueb/legal-bert-base-uncased"

TRAIN_DATA_PATH = "/content/task1.jsonl"
TEST_DATA_PATH = "/content/task_1_statute_prediction.jsonl"


In [ ]:
import os
import json

# Check whether both datasets exist
for file_path in [TRAIN_DATA_PATH, TEST_DATA_PATH]:
    if not os.path.exists(file_path):
        try:
            from google.colab import files

            print(f"{os.path.basename(file_path)} not found. Please upload it.")
            uploaded = files.upload()

            uploaded_name = list(uploaded.keys())[0]
            os.rename(uploaded_name, file_path)

        except ImportError:
            raise FileNotFoundError(
                f"Place {os.path.basename(file_path)} at {file_path} and re-run this cell."
            )


def load_json_or_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        text = f.read().strip()

    # Try normal JSON first
    try:
        obj = json.loads(text)
        return [obj] if isinstance(obj, dict) else obj
    except json.JSONDecodeError:
        pass

    # Otherwise treat as JSONL
    records = []
    for i, line in enumerate(text.splitlines()):
        line = line.strip()
        if not line:
            continue
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError as e:
            raise ValueError(
                f"Could not parse line {i+1} of {path} as JSON or JSONL:\n{e}"
            )

    return records


# Load datasets
train_raw_data = load_json_or_jsonl(TRAIN_DATA_PATH)
test_raw_data = load_json_or_jsonl(TEST_DATA_PATH)

print("=" * 50)
print(f"Training dataset : {TRAIN_DATA_PATH}")
print(f"Training records : {len(train_raw_data)}")
print("-" * 50)
print(f"Testing dataset  : {TEST_DATA_PATH}")
print(f"Testing records  : {len(test_raw_data)}")
print("=" * 50)
print("Train keys:", train_raw_data[0].keys())
print("Test keys :", test_raw_data[0].keys())

In [ ]:
from collections import defaultdict
import re


def normalize_section(s):
    """Convert '147', 'Section 147 IPC', 'IPC 147' -> 'IPC 147'"""
    if s is None:
        return ""

    m = re.search(r"(\d+[A-Za-z]?)", str(s))
    return f"IPC {m.group(1)}" if m else str(s).strip()


def parse_record(rec, is_test=False):
    fact = rec.get("fact", "").strip()

    # ----------------------------
    # Test dataset (prediction set)
    # ----------------------------
    if is_test:
        return {
            "doc_id": rec.get("doc_id", rec.get("id")),
            "fact": fact,
            "original_record": rec,      # Preserve the original JSON record
            "sentences": [],
            "sent2section": {},
            "gold_sections": [],
            "evidence_by_section": {},
            "gold_reasoning_by_section": {},
        }

    # ----------------------------
    # Training dataset
    # ----------------------------
    explanation = rec.get("explanation", {}) or {}

    gold_sections = sorted({
        normalize_section(s)
        for s in rec.get("statute", [])
    })

    # sentence -> IPC section
    sent2section = {
        sent.strip(): normalize_section(sec)
        for sent, sec in explanation.items()
    }

    # IPC section -> supporting sentences
    evidence_by_section = defaultdict(list)
    for sent, sec in sent2section.items():
        evidence_by_section[sec].append(sent)

    # Gold reasoning (if available)
    gold_reasoning_by_section = {}

    for sec in gold_sections:
        if (
            GOLD_REASONING_KEY in rec
            and isinstance(rec[GOLD_REASONING_KEY], dict)
            and sec in rec[GOLD_REASONING_KEY]
        ):
            gold_reasoning_by_section[sec] = rec[GOLD_REASONING_KEY][sec]
        else:
            gold_reasoning_by_section[sec] = (
                " ".join(evidence_by_section.get(sec, []))
                or fact[:200]
            )

    return {
        "doc_id": rec.get("doc_id", rec.get("id")),
        "fact": fact,
        "sentences": list(sent2section.keys()),
        "sent2section": sent2section,
        "gold_sections": gold_sections,
        "evidence_by_section": dict(evidence_by_section),
        "gold_reasoning_by_section": gold_reasoning_by_section,
    }


# ==========================================================
# Parse the training dataset
# ==========================================================

train_data_all = [
    parse_record(record)
    for record in train_raw_data
    if record.get("fact")
]

# Keep only labeled examples
train_data_all = [
    record
    for record in train_data_all
    if len(record["gold_sections"]) > 0
]


# ==========================================================
# Parse the prediction dataset
# ==========================================================

test_data = [
    parse_record(record, is_test=True)
    for record in test_raw_data
    if record.get("fact")
]


print("=" * 50)
print(f"Training samples : {len(train_data_all)}")
print(f"Testing samples  : {len(test_data)}")
print("=" * 50)

In [ ]:
from sklearn.model_selection import train_test_split

# ==========================================================
# Split the training dataset into Train / Validation
# ==========================================================

train_data, val_data = train_test_split(
    train_data_all,
    test_size=0.10,
    random_state=SEED,
    shuffle=True,
)

print("=" * 60)
print(f"Training samples   : {len(train_data)}")
print(f"Validation samples : {len(val_data)}")
print(f"Testing samples    : {len(test_data)}")
print("=" * 60)

# ==========================================================
# Build IPC vocabulary from the ENTIRE training corpus
# ==========================================================

SECTION_VOCAB = sorted({
    section
    for record in train_data_all
    for section in record["gold_sections"]
})

print(f"Number of IPC Sections : {len(SECTION_VOCAB)}")
print("First 20 Sections      :", SECTION_VOCAB[:20])

# Quick sanity check
assert len(train_data) + len(val_data) == len(train_data_all)
assert len(SECTION_VOCAB) > 0

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline,
)
import torch

# ==========================================================
# Load Tokenizer
# ==========================================================

tokenizer = AutoTokenizer.from_pretrained(
    LLM_MODEL_NAME,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ==========================================================
# Load LLM
#   GPU  -> 4-bit (nf4) so a 7B model fits comfortably on a free T4.
#   CPU  -> full precision fallback.
# ==========================================================

if DEVICE == "cuda":
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    llm_model = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
else:
    llm_model = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL_NAME,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    ).to(DEVICE)

# ==========================================================
# Text Generation Pipeline
# ==========================================================

gen_pipe = pipeline(
    "text-generation",
    model=llm_model,
    tokenizer=tokenizer,
    max_new_tokens=700,          # room for JSON: sections + evidence + traces
    do_sample=False,
    return_full_text=False,
    pad_token_id=tokenizer.eos_token_id,
)

In [ ]:
import json, re, difflib

# ==========================================================
# Prompt  (strict JSON, OPEN label space — not capped to 7)
# ==========================================================

SYSTEM_PROMPT = """You are an expert in Indian criminal law (Indian Penal Code).

Given the FACTS of a criminal case, identify ALL applicable IPC sections.
There is NO fixed list of sections. Predict every section the facts support.

Return ONLY a single valid JSON object, with no text before or after it,
using EXACTLY this schema:

{
  "sections": [
    {
      "section": "IPC <number>",
      "evidence": "<one sentence copied WORD-FOR-WORD from the FACTS>",
      "reasoning": "<why that sentence establishes this section>"
    }
  ],
  "reasoning_traces": "<step-by-step legal analysis linking the facts to the predicted sections>"
}

Rules:
- Copy each evidence sentence EXACTLY from the FACTS. Never paraphrase or shorten it.
- Do not invent facts that are not in the FACTS.
- Predict between 1 and 5 sections.
- Output the JSON object only."""


def _norm(sec):
    return normalize_section(sec)


def build_few_shot_block(examples, k=3):
    """Build a few JSON exemplars straight from the gold training data."""
    examples = random.Random(SEED).sample(examples, min(k, len(examples)))
    blocks = []
    for ex in examples:
        sections = []
        for sec in ex["gold_sections"][:5]:
            ev = ex["evidence_by_section"].get(sec, [ex["fact"][:150]])[0]
            rs = ex["gold_reasoning_by_section"].get(sec, "")[:160]
            sections.append({
                "section": sec,
                "evidence": ev,
                "reasoning": rs or f"The facts establish the ingredients of {sec}.",
            })
        target = {
            "sections": sections,
            "reasoning_traces": (
                "The facts were analysed and matched to the ingredients of "
                + ", ".join(ex["gold_sections"]) + "."
            ),
        }
        blocks.append(
            f"FACTS:\n{ex['fact'][:600]}\n\n"
            f"OUTPUT:\n{json.dumps(target, ensure_ascii=False)}"
        )
    return "\n\n".join(blocks)


FEW_SHOT_BLOCK = build_few_shot_block(train_data_all, k=3)


def make_prompt(fact_text, few_shot=False):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    if few_shot:
        messages.append({
            "role": "user",
            "content": "Here are solved examples:\n\n" + FEW_SHOT_BLOCK,
        })
        messages.append({"role": "assistant", "content": "Understood. I will output JSON only."})
    messages.append({"role": "user", "content": f"FACTS:\n{fact_text[:2800]}"})
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


# ==========================================================
# Verbatim enforcement: snap any evidence back to a real fact sentence
# ==========================================================

_SENT_SPLIT = re.compile(r"(?<=[.;])\s+(?=[A-Z0-9(])")


def split_into_sentences(text):
    parts = _SENT_SPLIT.split(text)
    return [p.strip() for p in parts if len(p.strip()) > 0]


def snap_to_fact(evidence, fact, fact_sents):
    """Return a sentence that is guaranteed to appear verbatim in `fact`."""
    if not evidence:
        return ""
    if evidence in fact:            # already verbatim
        return evidence
    if not fact_sents:
        return ""
    return max(
        fact_sents,
        key=lambda s: difflib.SequenceMatcher(None, s.lower(), evidence.lower()).ratio(),
    )


# ==========================================================
# Robust JSON parsing of the model output
# ==========================================================

def _extract_json(text):
    start = text.find("{")
    if start == -1:
        return None
    depth = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                chunk = text[start:i + 1]
                try:
                    return json.loads(chunk)
                except json.JSONDecodeError:
                    return None
    return None


def parse_llm_output(text, fact):
    """Parse JSON -> (predictions, reasoning_traces) with verbatim evidence."""
    fact_sents = split_into_sentences(fact)
    obj = _extract_json(text)

    predictions = []
    reasoning_trace = ""

    if isinstance(obj, dict):
        for item in obj.get("sections", []):
            if not isinstance(item, dict):
                continue
            sec = _norm(item.get("section", ""))
            if not sec:
                continue
            ev = snap_to_fact(str(item.get("evidence", "")).strip(), fact, fact_sents)
            predictions.append({
                "section": sec,
                "evidence": ev,
                "reasoning": str(item.get("reasoning", "")).strip(),
            })
        reasoning_trace = str(obj.get("reasoning_traces", "")).strip()

    # de-duplicate identical (section, evidence) pairs, keep order
    seen, deduped = set(), []
    for p in predictions:
        key = (p["section"], p["evidence"])
        if key not in seen:
            seen.add(key)
            deduped.append(p)
    return deduped[:5], reasoning_trace


def run_llm_baseline(dataset, few_shot=False, verbose_first_n=1):
    results = []
    for i, sample in enumerate(
        tqdm(dataset, desc=f"{'Few' if few_shot else 'Zero'}-shot inference")
    ):
        prompt = make_prompt(sample["fact"], few_shot=few_shot)
        output = gen_pipe(prompt)[0]["generated_text"]
        predictions, reasoning_trace = parse_llm_output(output, sample["fact"])

        if i < verbose_first_n:
            print("=" * 80)
            print(f"Document : {sample['doc_id']}")
            print(output[:1200])
            print("-> parsed sections:", [p["section"] for p in predictions])
            print("=" * 80)

        results.append({
            "doc_id": sample["doc_id"],
            "fact": sample["fact"],
            "predictions": predictions,
            "reasoning_traces": reasoning_trace,
        })
    return results


zero_shot_results = run_llm_baseline(test_data, few_shot=False)
few_shot_results = run_llm_baseline(test_data, few_shot=True)

In [ ]:
from transformers import (
    AutoTokenizer as BertTok,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from torch.utils.data import Dataset
import torch
import numpy as np

# ==========================================================
# Multi-label binarizer
# ==========================================================

mlb = MultiLabelBinarizer(classes=SECTION_VOCAB)
mlb.fit([SECTION_VOCAB])

# ==========================================================
# LegalBERT Tokenizer
# ==========================================================

bert_tokenizer = BertTok.from_pretrained(BERT_MODEL_NAME)


# ==========================================================
# Dataset
# ==========================================================

class IPCDataset(Dataset):
    def __init__(self, records, tokenizer, max_len=512, training=True):

        self.training = training
        self.tokenizer = tokenizer
        self.max_len = max_len

        if training:
            # Keep only labelled records
            self.records = [
                r for r in records
                if len(r["gold_sections"]) > 0
            ]
        else:
            # Test set has no gold labels
            self.records = records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):

        record = self.records[idx]

        enc = self.tokenizer(
            record["fact"],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
        )

        item = {
            key: torch.tensor(value)
            for key, value in enc.items()
        }

        if self.training:
            labels = mlb.transform(
                [record["gold_sections"]]
            )[0].astype(np.float32)

            item["labels"] = torch.tensor(labels)

        return item


# ==========================================================
# Build datasets
# ==========================================================

train_ds = IPCDataset(
    train_data,
    bert_tokenizer,
    training=True,
)

val_ds = IPCDataset(
    val_data,
    bert_tokenizer,
    training=True,
)

test_ds = IPCDataset(
    test_data,
    bert_tokenizer,
    training=False,
)


# ==========================================================
# LegalBERT classifier
# ==========================================================

bert_clf = AutoModelForSequenceClassification.from_pretrained(
    BERT_MODEL_NAME,
    num_labels=len(SECTION_VOCAB),
    problem_type="multi_label_classification",
)


# ==========================================================
# Metrics (validation only)
# ==========================================================

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    probs = 1 / (1 + np.exp(-logits))

    preds = (probs > 0.5).astype(int)

    return {
        "macro_f1": f1_score(
            labels,
            preds,
            average="macro",
            zero_division=0,
        )
    }


# ==========================================================
# Training configuration
# ==========================================================

training_args = TrainingArguments(
    output_dir="/content/bert_ipc",
    num_train_epochs=4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=20,
    learning_rate=2e-5,
    report_to=[],
    seed=SEED,
)


# ==========================================================
# Trainer
# ==========================================================

trainer = Trainer(
    model=bert_clf,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)


# ==========================================================
# Train
# ==========================================================

if len(train_ds) >= 8:

    trainer.train()

else:

    print(
        f"Only {len(train_ds)} training examples found. "
        "Skipping BERT fine-tuning."
    )

In [ ]:
def run_bert_baseline(dataset, top_k=5, threshold=0.5):
    """
    Run LegalBERT on the prediction dataset.

    Returns:
        results: list of dictionaries containing
            - doc_id
            - fact
            - predicted_sections
            - ranked_sections
            - probabilities
    """

    results = []

    bert_clf.eval()
    bert_clf.to(DEVICE)

    with torch.no_grad():

        for sample in tqdm(dataset, desc="BERT inference"):

            enc = bert_tokenizer(
                sample["fact"],
                truncation=True,
                padding="max_length",
                max_length=512,
                return_tensors="pt",
            ).to(DEVICE)

            logits = bert_clf(**enc).logits.squeeze(0)

            probs = torch.sigmoid(logits).cpu().numpy()

            order = np.argsort(-probs)

            ranked_sections = [
                SECTION_VOCAB[i]
                for i in order
            ]

            ranked_scores = [
                float(probs[i])
                for i in order
            ]

            predicted_sections = [
                SECTION_VOCAB[i]
                for i in order
                if probs[i] >= threshold
            ]

            # Always predict at least one IPC section
            if len(predicted_sections) == 0:
                predicted_sections = ranked_sections[:1]

            predicted_sections = predicted_sections[:top_k]

            predictions = []

            for section in predicted_sections:
                predictions.append(
                    {
                        "section": section,
                        "evidence": "",
                        "reasoning": "",
                    }
                )

            results.append(
                {
                    "doc_id": sample["doc_id"],
                    "fact": sample["fact"],
                    "predictions": predictions,
                    "ranked_sections": ranked_sections,
                    "probabilities": ranked_scores,
                }
            )

    return results


# --------------------------------------------------------
# Run inference
# --------------------------------------------------------

if len(train_ds) >= 8:
    bert_results = run_bert_baseline(test_data)
else:
    bert_results = []
    print("BERT model was not trained. Skipping inference.")

In [ ]:
# ==========================================================
# Prediction Summary
# ==========================================================

def summarize_predictions(name, results):

    if not results:
        return None

    num_predictions = [
        len(r["predictions"])
        for r in results
    ]

    summary = {
        "Model": name,
        "Cases": len(results),
        "Average IPCs Predicted": round(np.mean(num_predictions), 2),
        "Minimum IPCs": min(num_predictions),
        "Maximum IPCs": max(num_predictions),
    }

    return summary


summary = [
    summarize_predictions(
        "Zero-shot LLM",
        zero_shot_results
    ),
    summarize_predictions(
        "Few-shot LLM",
        few_shot_results
    ),
    summarize_predictions(
        "LegalBERT",
        bert_results
    ),
]

summary = [s for s in summary if s is not None]

df = pd.DataFrame(summary)

print(df)

df

In [ ]:
import json
from collections import defaultdict

# ==========================================================
# Convert predictions to competition format
# ==========================================================

def predictions_to_explanation(predictions):
    """[{section, evidence, ...}, ...] -> {evidence_sentence: [sections]}"""
    explanation = defaultdict(list)
    for pred in predictions:
        evidence = pred.get("evidence", "").strip()
        section = normalize_section(pred.get("section", ""))
        if evidence and section:
            if section not in explanation[evidence]:
                explanation[evidence].append(section)
    return dict(explanation)


def synthesize_trace(predictions):
    """Build a real reasoning trace from the predictions (never generic boilerplate)."""
    if not predictions:
        return "No applicable IPC section could be identified from the facts."
    lines = ["The following IPC sections were identified from the facts:"]
    for i, p in enumerate(predictions, 1):
        sec = normalize_section(p.get("section", ""))
        ev = p.get("evidence", "").strip()
        rs = p.get("reasoning", "").strip()
        piece = f'{i}. {sec}'
        if ev:
            piece += f' — supported by: "{ev}"'
        if rs:
            piece += f' ({rs})'
        lines.append(piece)
    return "\n".join(lines)


# ==========================================================
# Create submission JSONL
# ==========================================================

# Exact schema of the test set (drives strict enforcement below)
SUBMISSION_KEYS = ["id", "fact", "reasoning_traces", "explanation"]


def coerce_row(row):
    """Force a row into the EXACT test-set schema:
       id:str, fact:str, reasoning_traces:str, explanation:dict[str, list[str]]."""
    exp = row.get("explanation", {})
    clean_exp = {}
    if isinstance(exp, dict):
        for sent, secs in exp.items():
            sent = str(sent)
            if isinstance(secs, (list, tuple)):
                secs = [str(s) for s in secs]
            else:                         # tolerate a bare string value
                secs = [str(secs)]
            clean_exp[sent] = secs
    # rebuild with the exact key order, dropping any extra keys
    return {
        "id": str(row.get("id", "")),
        "fact": str(row.get("fact", "")),
        "reasoning_traces": str(row.get("reasoning_traces") or ""),
        "explanation": clean_exp,
    }


def create_submission(results, original_test_data, output_file="submission.jsonl"):
    submission = []
    for sample, prediction in zip(original_test_data, results):
        reasoning = prediction.get("reasoning_traces", "").strip()
        if not reasoning:
            reasoning = synthesize_trace(prediction["predictions"])

        explanation = predictions_to_explanation(prediction["predictions"])

        submission.append(coerce_row({
            "id": sample["original_record"].get("id", sample["doc_id"]),
            "fact": sample["fact"],
            "reasoning_traces": reasoning,
            "explanation": explanation,
        }))

    with open(output_file, "w", encoding="utf-8") as f:
        for row in submission:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    print("=" * 60)
    print(f"Saved {len(submission)} predictions")
    print(f"Output file : {output_file}")
    print("=" * 60)
    return submission


# ==========================================================
# Choose which model to submit  (few-shot LLM = open label space)
# ==========================================================

submission = create_submission(
    few_shot_results,      # change to zero_shot_results or bert_results if desired
    test_data,
    output_file="submission.jsonl",
)

# ---- Quick quality report ----
import collections
verb = sum(
    1 for r in submission for ev in r["explanation"] if ev in r["fact"]
)
total_ev = sum(len(r["explanation"]) for r in submission)
sec_counts = collections.Counter(
    s for r in submission for secs in r["explanation"].values() for s in secs
)
print(f"Verbatim evidence: {verb}/{total_ev}")
print(f"Distinct sections predicted: {len(sec_counts)}")
print("Section distribution:", dict(sec_counts.most_common()))

print(json.dumps(submission[0], indent=4, ensure_ascii=False))

In [ ]:
# ==========================================================
# STRICT format enforcement — submission must match the test schema
#   id:str, fact:str, reasoning_traces:str, explanation:dict[str, list[str]]
# ==========================================================

def validate_submission(submission, test_records, output_file="submission.jsonl"):
    errors = []

    # 1) same number of rows, aligned to the test set
    if len(submission) != len(test_records):
        errors.append(
            f"row count {len(submission)} != test count {len(test_records)}"
        )

    test_ids = [r.get("id") for r in test_records]

    for idx, row in enumerate(submission):
        tag = f"row {idx}"

        # 2) exact keys, exact order
        if list(row.keys()) != SUBMISSION_KEYS:
            errors.append(f"{tag}: keys {list(row.keys())} != {SUBMISSION_KEYS}")
            continue

        # 3) types
        if not isinstance(row["id"], str):
            errors.append(f"{tag}: id not str")
        if not isinstance(row["fact"], str):
            errors.append(f"{tag}: fact not str")
        if not isinstance(row["reasoning_traces"], str):
            errors.append(f"{tag}: reasoning_traces not str")
        if not isinstance(row["explanation"], dict):
            errors.append(f"{tag}: explanation not dict")
            continue

        # 4) explanation must be dict[str, list[str]]
        for sent, secs in row["explanation"].items():
            if not isinstance(sent, str):
                errors.append(f"{tag}: explanation key not str")
            if not isinstance(secs, list) or not all(isinstance(s, str) for s in secs):
                errors.append(f"{tag}: explanation['{sent[:30]}...'] not list[str]")

        # 5) id must line up with the test record at the same position
        if idx < len(test_ids) and row["id"] != test_ids[idx]:
            errors.append(f"{tag}: id '{row['id']}' != test id '{test_ids[idx]}'")

    # 6) the written file must be valid JSONL with the same schema
    for i, line in enumerate(open(output_file, encoding="utf-8")):
        line = line.strip()
        if not line:
            continue
        try:
            obj = json.loads(line)
        except json.JSONDecodeError as e:
            errors.append(f"file line {i+1}: invalid JSON ({e})")
            continue
        if list(obj.keys()) != SUBMISSION_KEYS:
            errors.append(f"file line {i+1}: keys {list(obj.keys())} != {SUBMISSION_KEYS}")

    if errors:
        print("FORMAT CHECK FAILED:")
        for e in errors[:30]:
            print("  -", e)
        raise AssertionError(f"{len(errors)} submission format error(s).")

    print("=" * 60)
    print(f"FORMAT OK — {len(submission)} rows match the test schema exactly:")
    print("  keys        :", SUBMISSION_KEYS)
    print("  explanation : dict[str, list[str]]")
    print(f"  ids aligned : {submission[0]['id']} ... {submission[-1]['id']}")
    print("=" * 60)


validate_submission(submission, test_raw_data, "submission.jsonl")